In [3]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-22 20:54:21--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-03-22 20:54:22 (22.1 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [4]:
import torch

In [5]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print('length of dataset in characters: ', len(text))

length of dataset in characters:  1115394


In [7]:
# let's looks at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [8]:
# Building Vocab
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [9]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a stringArithmeticError

# This is a character level tokenizer
# Google uses sentencepiece tokenizer.
# OpenAI uses tiktoken 

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [10]:
# let's now encode the enitre text dataset and store it into a torch.Tensor
import torch # we use PyTorch
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
# print(data[:1000]) # the first 1000 characters in numerical tensor form

torch.Size([1115394]) torch.int64


In [11]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [12]:
print(train_data.shape)
type(train_data)

torch.Size([1003854])


torch.Tensor

In [13]:
# Context Window
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [14]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target: {target}')

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [15]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # maximum context length for predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # We grab 4 (batch_size) random index values from the corpus and create the batch from them.
    # So suppose we find the index n, we get the first training example from n to n+8 (block_size=8)
    # And we create a +1 offset for the target.

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension (num_tokens)
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target : {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target : 43
when input is [24, 43] the target : 58
when input is [24, 43, 58] the target : 5
when input is [24, 43, 58, 5] the target : 57
when input is [24, 43, 58, 5, 57] the target : 1
when input is [24, 43, 58, 5, 57, 1] the target : 46
when input is [24, 43, 58, 5, 57, 1, 46] the target : 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target : 39
when input is [44] the target : 53
when input is [44, 53] the target : 56
when input is [44, 53, 56] the target : 1
when input is [44, 53, 56, 1] the target : 58
when input is [44, 53, 56, 1, 58] the target : 46
when inp

In [16]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from the lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C)
        # Dimension: (Batch, Time, Embedding)

        # Now, the cross_entropy loss function in pytorch expects the Channel Dimension (Embedding Dimension) 
        # to be at the 2nd dimension in the tensor. 
        # Therefore, we will reshape our logits to merge the Batch and Time dimension to bring C to the 2nd dimension 

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # Reshaping our Tensors
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in current context
        for _ in range(max_new_tokens):

            # get the predictions
            logits, loss = self(idx) # We are calling the forward function of the model
            # logits Dimension: (Batch_size, Time, Logits)

            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # Dimension: (Batch_size, Logits)

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Picking the index of with probability directly proportional to the value at the index
            # Dimension: (Batch_size, 1)

            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            
        return idx

In [24]:
model = BigramLanguageModel(vocab_size)
model

BigramLanguageModel(
  (token_embedding_table): Embedding(65, 65)
)

In [25]:
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [47]:
idx = torch.zeros((1, 1), dtype=torch.long)
# Creating an initial input with dimension: (Batch_Size, Time {Num_Tokens}) 

output = model.generate(idx, max_new_tokens=100)
# Calling generate function
print(output.shape)

print(decode(output[0].tolist()))
# printing the first 

torch.Size([1, 101])

E:Cgxld,SPP -ZM&sBBN&llvTEeZ.kci!oXxfgID.ixWKbOL&uR.JBLA-bnQ!
VrgCJGJGSYdcriBq$WGlvniGS fybjR:wD -Cv


In [49]:
output = model.generate(idx = torch.zeros((4, 1), dtype=torch.long), max_new_tokens=100)
# Passing batch_size as 4 and getting 4 batch's predictions in parallel
output.shape

torch.Size([4, 101])

In [50]:
# Decoding all 4 batches
for i in range(4):
    print(decode(output[i].tolist()))



pBL:'qG,JaZcLtQVptXEQ$VzbYiOLaXecoiP -UGbrIcRhotxIQA'kYdmCllxfIiFBHRJdERlTZ;aYho'X',pwUMbxJMkzIMy;JD

lgVzapC&yevYHASZ?KHJ$eQGX:UQOw'aoh,ehPiXY qgmIQHALI&WWNjngmvELhO:MkgCgbzRjDzP!AcoXEtR?Y!R' 'W,liK$V&

NrWzm3:3QdubbdZ;?$DhlI&!MszLMlgQzmZd,q,.mmmyhVzW:Cj?rVBoZvWb;:CZ;--Y-y;.D.j'hSZJ;
Vz,Ac' W?uAQbQwxly

?YNoiXFOpAztUoSKRw'uxKu3R
o'xBNDxJdEG,qRqboCXEpoD&aPKuGU;Ar3SfR?a!QJAaVPRqXrW
$zrzrZ?GC zs$
I,CjKJ$G


torch.Size([1, 101])

KwA-Kc!o?,ZS&TDylZAcPiqZYC 3QXiz fXehVbOmIlyhsVTyd?o.?a!AuUluCjpOi.E3flNA;a!w3KZxOsHo?:ztoiBjRpJqFq:
